**WEEK 1: DATA INGESTION**

In [ ]:
# Install PDF and image processing packages
!pip install -q pdfplumber pytesseract pillow pandas numpy pdf2image
!apt-get install -q tesseract-ocr poppler-utils

# Verify installations
print("\n✅ All packages installed successfully!")
print("\n📦 Installed components:")
print("  • pdfplumber - PDF text extraction")
print("  • pytesseract - OCR for images")
print("  • pdf2image - Convert PDF pages to images")
print("  • poppler-utils - PDF rendering engine")
print("  • tesseract-ocr - OCR engine")
print("\n")

In [ ]:
print("IMPORTING LIBRARIES")


import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pandas as pd
import numpy as np
import re
import json
import io
import os
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from enum import Enum
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("\n✅ All libraries imported successfully!")
print("\n")

In [ ]:
print("SETTING UP DATA STRUCTURES")

class ParameterStatus(Enum):
    """Classification status for blood parameters"""
    NORMAL = "Normal"
    HIGH = "High"
    LOW = "Low"
    BORDERLINE_HIGH = "Borderline High"
    BORDERLINE_LOW = "Borderline Low"
    CRITICAL_HIGH = "Critical High"
    CRITICAL_LOW = "Critical Low"

@dataclass
class BloodParameter:
    """Data structure for blood test parameter"""
    name: str
    value: float
    unit: str
    reference_min: float
    reference_max: float
    status: ParameterStatus
    raw_text: str = ""

@dataclass
class ExtractionResult:
    """Result of data extraction from blood report"""
    parameters: List[BloodParameter]
    raw_text: str
    extraction_method: str
    success: bool
    errors: List[str]

print("\n✅ Data structures configured!")
print("\n")

In [ ]:
print("🩺 LOADING MEDICAL REFERENCE RANGES")

REFERENCE_RANGES = {
    # Complete Blood Count (CBC)
    'hemoglobin': {
        'unit': 'g/dL',
        'male': (13.5, 17.5),
        'female': (12.0, 15.5),
        'default': (12.0, 17.5),
        'critical_low': 7.0,
        'critical_high': 20.0,
        'aliases': ['hb', 'hgb', 'haemoglobin', 'hemoglobin level']
    },
    'rbc': {
        'unit': 'million cells/µL',
        'male': (4.5, 5.9),
        'female': (4.0, 5.2),
        'default': (4.0, 5.9),
        'critical_low': 2.5,
        'critical_high': 7.0,
        'aliases': ['red blood cell count', 'red blood cells', 'erythrocytes', 'rbc count']
    },
    'wbc': {
        'unit': 'thousand cells/µL',
        'default': (4.0, 11.0),
        'critical_low': 2.0,
        'critical_high': 30.0,
        'aliases': ['white blood cell count', 'white blood cells', 'leukocytes', 'total leucocyte count', 'tlc', 'wbc count']
    },
    'platelets': {
        'unit': 'thousand cells/µL',
        'default': (150, 400),
        'critical_low': 50,
        'critical_high': 1000,
        'aliases': ['platelet count', 'thrombocytes', 'platelet']
    },
    'hematocrit': {
        'unit': '%',
        'male': (38.8, 50.0),
        'female': (34.9, 44.5),
        'default': (34.9, 50.0),
        'critical_low': 20.0,
        'critical_high': 60.0,
        'aliases': ['hct', 'packed cell volume', 'pcv']
    },
    'mcv': {
        'unit': 'fL',
        'default': (80, 100),
        'critical_low': 60,
        'critical_high': 120,
        'aliases': ['mean corpuscular volume', 'mean cell volume']
    },
    'mch': {
        'unit': 'pg',
        'default': (27, 33),
        'critical_low': 20,
        'critical_high': 40,
        'aliases': ['mean corpuscular hemoglobin', 'mean cell hemoglobin']
    },
    'mchc': {
        'unit': 'g/dL',
        'default': (32, 36),
        'critical_low': 28,
        'critical_high': 40,
        'aliases': ['mean corpuscular hemoglobin concentration']
    },

    # Glucose & Diabetes
    'glucose': {
        'unit': 'mg/dL',
        'fasting': (70, 100),
        'random': (70, 140),
        'default': (70, 140),
        'critical_low': 40,
        'critical_high': 400,
        'aliases': ['blood sugar', 'blood glucose', 'fbs', 'fasting blood sugar', 'rbs', 'random blood sugar', 'plasma glucose']
    },
    'hba1c': {
        'unit': '%',
        'default': (4.0, 5.6),
        'critical_low': 3.0,
        'critical_high': 15.0,
        'aliases': ['glycated hemoglobin', 'glycosylated hemoglobin', 'a1c', 'glycated hb']
    },

    # Lipid Profile
    'cholesterol': {
        'unit': 'mg/dL',
        'default': (0, 200),
        'critical_low': 0,
        'critical_high': 400,
        'aliases': ['total cholesterol', 'chol', 'serum cholesterol']
    },
    'hdl': {
        'unit': 'mg/dL',
        'male': (40, 999),
        'female': (50, 999),
        'default': (40, 999),
        'critical_low': 20,
        'critical_high': 100,
        'aliases': ['hdl cholesterol', 'good cholesterol', 'high density lipoprotein']
    },
    'ldl': {
        'unit': 'mg/dL',
        'default': (0, 100),
        'critical_low': 0,
        'critical_high': 300,
        'aliases': ['ldl cholesterol', 'bad cholesterol', 'low density lipoprotein']
    },
    'triglycerides': {
        'unit': 'mg/dL',
        'default': (0, 150),
        'critical_low': 0,
        'critical_high': 1000,
        'aliases': ['tg', 'trigs', 'triglyceride']
    },

    # Kidney Function
    'creatinine': {
        'unit': 'mg/dL',
        'male': (0.7, 1.3),
        'female': (0.6, 1.1),
        'default': (0.6, 1.3),
        'critical_low': 0.3,
        'critical_high': 10.0,
        'aliases': ['serum creatinine', 'creat', 's.creatinine']
    },
    'bun': {
        'unit': 'mg/dL',
        'default': (7, 20),
        'critical_low': 2,
        'critical_high': 100,
        'aliases': ['blood urea nitrogen', 'urea nitrogen', 'urea']
    },
    'uric_acid': {
        'unit': 'mg/dL',
        'male': (3.4, 7.0),
        'female': (2.4, 6.0),
        'default': (2.4, 7.0),
        'critical_low': 1.0,
        'critical_high': 15.0,
        'aliases': ['uric acid', 'urate', 'serum uric acid']
    },

    # Liver Function
    'alt': {
        'unit': 'U/L',
        'default': (7, 56),
        'critical_low': 0,
        'critical_high': 1000,
        'aliases': ['sgpt', 'alanine aminotransferase', 'alanine transaminase', 'alat']
    },
    'ast': {
        'unit': 'U/L',
        'default': (10, 40),
        'critical_low': 0,
        'critical_high': 1000,
        'aliases': ['sgot', 'aspartate aminotransferase', 'aspartate transaminase', 'asat']
    },
    'alp': {
        'unit': 'U/L',
        'default': (44, 147),
        'critical_low': 0,
        'critical_high': 1000,
        'aliases': ['alkaline phosphatase', 'alk phos']
    },
    'bilirubin_total': {
        'unit': 'mg/dL',
        'default': (0.1, 1.2),
        'critical_low': 0,
        'critical_high': 20.0,
        'aliases': ['total bilirubin', 'bilirubin', 't.bilirubin']
    },
    'bilirubin_direct': {
        'unit': 'mg/dL',
        'default': (0.0, 0.3),
        'critical_low': 0,
        'critical_high': 10.0,
        'aliases': ['direct bilirubin', 'conjugated bilirubin', 'd.bilirubin']
    },
    'albumin': {
        'unit': 'g/dL',
        'default': (3.5, 5.5),
        'critical_low': 2.0,
        'critical_high': 6.0,
        'aliases': ['serum albumin', 's.albumin']
    },
    'total_protein': {
        'unit': 'g/dL',
        'default': (6.0, 8.3),
        'critical_low': 4.0,
        'critical_high': 10.0,
        'aliases': ['protein total', 'serum protein', 'total proteins']
    },

    # Thyroid Function
    'tsh': {
        'unit': 'mIU/L',
        'default': (0.4, 4.0),
        'critical_low': 0.01,
        'critical_high': 100.0,
        'aliases': ['thyroid stimulating hormone', 'thyrotropin', 'sensitive tsh', 'ultrasensitive tsh']
    },
    't3': {
        'unit': 'ng/dL',
        'default': (80, 200),
        'critical_low': 40,
        'critical_high': 400,
        'aliases': ['triiodothyronine', 'total t3', 't3 total']
    },
    't4': {
        'unit': 'µg/dL',
        'default': (5.0, 12.0),
        'critical_low': 2.0,
        'critical_high': 25.0,
        'aliases': ['thyroxine', 'total t4', 't4 total']
    },

    # Electrolytes
    'sodium': {
        'unit': 'mmol/L',
        'default': (136, 145),
        'critical_low': 120,
        'critical_high': 160,
        'aliases': ['na', 'serum sodium', 's.sodium']
    },
    'potassium': {
        'unit': 'mmol/L',
        'default': (3.5, 5.0),
        'critical_low': 2.5,
        'critical_high': 6.5,
        'aliases': ['k', 'serum potassium', 's.potassium']
    },
    'chloride': {
        'unit': 'mmol/L',
        'default': (96, 106),
        'critical_low': 80,
        'critical_high': 120,
        'aliases': ['cl', 'serum chloride', 's.chloride']
    },
    'calcium': {
        'unit': 'mg/dL',
        'default': (8.5, 10.5),
        'critical_low': 6.0,
        'critical_high': 13.0,
        'aliases': ['ca', 'serum calcium', 's.calcium', 'total calcium']
    }
}

# Unit conversion factors
UNIT_CONVERSIONS = {
    'hemoglobin': {
        'g/L': 0.1,
        'g/dl': 1.0,
        'g/dL': 1.0,
    },
    'glucose': {
        'mmol/L': 18.0,
        'mmol/l': 18.0,
        'mg/dl': 1.0,
        'mg/dL': 1.0,
    },
    'cholesterol': {
        'mmol/L': 38.67,
        'mmol/l': 38.67,
        'mg/dl': 1.0,
        'mg/dL': 1.0,
    },
    'hdl': {
        'mmol/L': 38.67,
        'mg/dl': 1.0,
        'mg/dL': 1.0,
    },
    'ldl': {
        'mmol/L': 38.67,
        'mg/dl': 1.0,
        'mg/dL': 1.0,
    },
    'triglycerides': {
        'mmol/L': 88.57,
        'mg/dl': 1.0,
        'mg/dL': 1.0,
    },
    'creatinine': {
        'µmol/L': 0.0113,
        'umol/L': 0.0113,
        'μmol/L': 0.0113,
        'mg/dl': 1.0,
        'mg/dL': 1.0,
    }
}

print(f"\n✅ Loaded {len(REFERENCE_RANGES)} blood parameters")
print("   Including: CBC, Metabolic Panel, Lipids, Kidney, Liver, Thyroid, Electrolytes")
print("\n")

In [ ]:
print("BUILDING UNIVERSAL INPUT PARSER")

import cv2
import numpy as np

class UniversalInputParser:
    """Handles ANY input format - PDFs, images, scanned docs, JSON, text"""

    @staticmethod
    def preprocess_for_ocr(pil_image):
        """
        Improve OCR accuracy using grayscale + adaptive thresholding
        """
        img = np.array(pil_image)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )[1]
        return Image.fromarray(gray)

    @staticmethod
    def parse_file(file_path: str):
        print(f"\n🔍 Analyzing file: {file_path}")
        extension = file_path.lower().split('.')[-1]

        if extension == 'json':
            return UniversalInputParser._parse_json_file(file_path)
        elif extension in ['png', 'jpg', 'jpeg', 'tiff', 'bmp']:
            return UniversalInputParser._parse_image_file(file_path)
        elif extension == 'pdf':
            return UniversalInputParser._parse_pdf_universal(file_path)
        else:
            return UniversalInputParser._parse_text_file(file_path)

    @staticmethod
    def _parse_pdf_universal(file_path: str):
        try:
            text = ""
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"

            if text.strip() and len(text) > 50:
                return text, True, "pdf_text_extraction"
        except:
            pass

        # OCR fallback
        images = convert_from_path(file_path, dpi=300)
        text = ""
        for image in images:
            image = UniversalInputParser.preprocess_for_ocr(image)
            text += pytesseract.image_to_string(image, lang="eng")

        return text, bool(text.strip()), "pdf_ocr"

    @staticmethod
    def _parse_image_file(file_path: str):
        image = Image.open(file_path)
        image = UniversalInputParser.preprocess_for_ocr(image)
        text = pytesseract.image_to_string(image, lang="eng")
        return text, bool(text.strip()), "image_ocr"

    @staticmethod
    def _parse_text_file(file_path: str):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read(), True, "text_file"

    @staticmethod
    def _parse_json_file(file_path: str):
        with open(file_path, "r", encoding="utf-8") as f:
            return json.dumps(json.load(f)), True, "json_file"

print("Universal parser with OCR preprocessing ready!")

**WEEK 2: PARAMETER INTERPRATATION**

In [ ]:
print("BUILDING DATA EXTRACTION ENGINE")

class DataExtractionEngine:
    """Extracts blood parameters from text or JSON"""

    def __init__(self):
        self.reference_ranges = REFERENCE_RANGES

    def extract(self, content: str, method: str):
        return self.extract_from_text(content)

    def extract_from_text(self, text: str):
        parameters = []
        errors = []
        lines = text.split("\n")

        for param_name, info in self.reference_ranges.items():
            search_terms = [param_name] + info.get("aliases", [])

            for line in lines:
                line_lower = line.lower()
                if any(term.lower() in line_lower for term in search_terms):
                    value, unit = self._extract_value_from_line(
                        line, param_name, info["unit"]
                    )
                    if value is not None:
                        ref_min, ref_max = self._get_reference_range(param_name)
                        parameters.append(
                            BloodParameter(
                                name=param_name,
                                value=value,
                                unit=info["unit"],
                                reference_min=ref_min,
                                reference_max=ref_max,
                                status=ParameterStatus.NORMAL,
                                raw_text=line.strip()
                            )
                        )
                        break

        return ExtractionResult(
            parameters=parameters,
            raw_text=text,
            extraction_method="text_parsing",
            success=bool(parameters),
            errors=errors
        )

    def _extract_value_from_line(self, line, param_name, expected_unit):
        patterns = [
            r'(\d+\.?\d*)\s*' + re.escape(expected_unit),
            r':\s*(\d+\.?\d*)',
            r'=\s*(\d+\.?\d*)',
            r'(\d+,\d+)',
        ]

        ref_min, ref_max = self._get_reference_range(param_name)

        for pattern in patterns:
            match = re.search(pattern, line, re.IGNORECASE)
            if match:
                try:
                    value = float(match.group(1).replace(",", "."))
                    # SANITY FILTER
                    if ref_min * 0.1 <= value <= ref_max * 5:
                        return value, expected_unit
                except ValueError:
                    pass
        return None, ""

    def _get_reference_range(self, param_name):
        info = self.reference_ranges[param_name]
        return info.get("default", (0, 999))

print("Data extraction with clinical sanity filtering ready!")

In [ ]:
print("BUILDING DATA VALIDATOR")

class DataValidator:
    """Validates and standardizes extracted blood parameters"""

    def __init__(self):
        self.reference_ranges = REFERENCE_RANGES
        self.unit_conversions = UNIT_CONVERSIONS

    def validate_and_standardize(self, extraction_result: ExtractionResult) -> ExtractionResult:
        """
        Validate and standardize all parameters
        """
        print(f"\n   🔍 Validating {len(extraction_result.parameters)} parameters...")

        validated_params = []
        errors = list(extraction_result.errors)

        for param in extraction_result.parameters:
            # Convert units if needed
            standardized_value = self._convert_unit(param.name, param.value, param.unit)

            if standardized_value is None:
                errors.append(f"Could not convert {param.name} from {param.unit}")
                continue

            # Check plausibility
            if not self._is_plausible(param.name, standardized_value):
                errors.append(f"Implausible value for {param.name}: {standardized_value}")
                continue

            # Update with standardized value
            param.value = standardized_value
            param.unit = self.reference_ranges[param.name]['unit']

            validated_params.append(param)

        print(f"   Validated {len(validated_params)} parameters")
        if errors:
            print(f"   ⚠️  {len(errors)} validation warnings")

        return ExtractionResult(
            parameters=validated_params,
            raw_text=extraction_result.raw_text,
            extraction_method=extraction_result.extraction_method,
            success=len(validated_params) > 0,
            errors=errors
        )

    def _convert_unit(self, param_name: str, value: float, unit: str) -> Optional[float]:
        """Convert value to standard unit"""
        standard_unit = self.reference_ranges[param_name]['unit']

        # Already in standard unit
        if unit.lower().replace(' ', '') == standard_unit.lower().replace(' ', ''):
            return value

        # Check conversion factors
        if param_name in self.unit_conversions:
            conversions = self.unit_conversions[param_name]
            for conv_unit, factor in conversions.items():
                if unit.lower().replace(' ', '') == conv_unit.lower().replace(' ', ''):
                    return value * factor

        # No conversion available, return original
        return value

    def _is_plausible(self, param_name: str, value: float) -> bool:
        """Check if value is within plausible biological range"""
        param_info = self.reference_ranges[param_name]

        critical_low = param_info.get('critical_low', 0)
        critical_high = param_info.get('critical_high', 999999)

        # Allow values within critical range
        return critical_low * 0.5 <= value <= critical_high * 1.5

print("\n Data validator ready!")
print("\n")

In [ ]:
print("🤖 BUILDING PARAMETER INTERPRETER (MODEL 1)")

class ParameterInterpreter:
    """Model 1: Interprets individual blood parameters"""

    def __init__(self):
        self.reference_ranges = REFERENCE_RANGES

    def classify_parameters(self, extraction_result: ExtractionResult) -> ExtractionResult:
        """
        Classify each parameter status
        """
        print(f"\n   🔬 Classifying {len(extraction_result.parameters)} parameters...")

        classified_params = []

        for param in extraction_result.parameters:
            status = self._classify_single_parameter(param)
            param.status = status
            classified_params.append(param)

        # Count by status
        status_counts = {}
        for param in classified_params:
            status = param.status.value
            status_counts[status] = status_counts.get(status, 0) + 1

        print(f"   ✅ Classification complete")
        for status, count in sorted(status_counts.items()):
            print(f"      • {status}: {count}")

        return ExtractionResult(
            parameters=classified_params,
            raw_text=extraction_result.raw_text,
            extraction_method=extraction_result.extraction_method,
            success=extraction_result.success,
            errors=extraction_result.errors
        )

    def _classify_single_parameter(self, param: BloodParameter) -> ParameterStatus:
        """Classify a single parameter"""
        value = param.value
        ref_min = param.reference_min
        ref_max = param.reference_max

        param_info = self.reference_ranges[param.name]
        critical_low = param_info.get('critical_low', 0)
        critical_high = param_info.get('critical_high', 999999)

        # Calculate borderline thresholds
        margin = 0.05
        range_span = ref_max - ref_min
        borderline_low_threshold = ref_min + (range_span * margin)
        borderline_high_threshold = ref_max - (range_span * margin)

        # Classify
        if value <= critical_low:
            return ParameterStatus.CRITICAL_LOW
        elif value >= critical_high:
            return ParameterStatus.CRITICAL_HIGH
        elif value < ref_min:
            if value < borderline_low_threshold:
                return ParameterStatus.LOW
            else:
                return ParameterStatus.BORDERLINE_LOW
        elif value > ref_max:
            if value > borderline_high_threshold:
                return ParameterStatus.HIGH
            else:
                return ParameterStatus.BORDERLINE_HIGH
        else:
            return ParameterStatus.NORMAL

    def generate_report(self, extraction_result: ExtractionResult) -> str:
        """Generate human-readable report"""
        report = []
        report.append()
        report.append("BLOOD REPORT ANALYSIS - COMPREHENSIVE RESULTS")
        report.append()
        report.append("")

        if not extraction_result.success:
            report.append("❌ ANALYSIS FAILED")
            report.append("")
            report.append("No valid parameters could be extracted from the report.")
            if extraction_result.errors:
                report.append("\nErrors encountered:")
                for error in extraction_result.errors:
                    report.append(f"  • {error}")
            return "\n".join(report)

        # Summary statistics
        status_counts = {}
        for param in extraction_result.parameters:
            status = param.status.value
            status_counts[status] = status_counts.get(status, 0) + 1

        report.append(f"📊 ANALYSIS SUMMARY")
        report.append()
        report.append(f"Total Parameters Analyzed: {len(extraction_result.parameters)}")
        report.append("")
        report.append("Status Distribution:")
        for status, count in sorted(status_counts.items()):
            emoji = self._get_status_emoji(status)
            report.append(f"  {emoji} {status}: {count}")
        report.append("")
        report.append()
        report.append("")

        # Group parameters by status
        critical = [p for p in extraction_result.parameters
                   if 'CRITICAL' in p.status.value]
        abnormal = [p for p in extraction_result.parameters
                   if p.status != ParameterStatus.NORMAL and 'CRITICAL' not in p.status.value]
        normal = [p for p in extraction_result.parameters
                 if p.status == ParameterStatus.NORMAL]

        # Critical results first
        if critical:
            report.append("🔴 CRITICAL RESULTS - IMMEDIATE ATTENTION REQUIRED")
            report.append()
            report.append("These values are outside safe ranges and may require urgent medical care.")
            report.append("")
            for param in critical:
                report.append(self._format_parameter(param))
            report.append("")

        # Abnormal results
        if abnormal:
            report.append("⚠️  ABNORMAL RESULTS")
            report.append()
            report.append("These values are outside normal ranges and should be discussed with")
            report.append("your healthcare provider.")
            report.append("")
            for param in abnormal:
                report.append(self._format_parameter(param))
            report.append("")

        # Normal results
        if normal:
            report.append("✅ NORMAL RESULTS")
            report.append()
            report.append("These values are within normal reference ranges.")
            report.append("")
            for param in normal:
                report.append(self._format_parameter(param))
            report.append("")

        # Warnings/Errors
        if extraction_result.errors:
            report.append("⚠️  WARNINGS & NOTES")
            report.append()
            for error in extraction_result.errors:
                report.append(f"  • {error}")
            report.append("")

        # Disclaimer
        report.append("=" * 80)
        report.append("⚕️  MEDICAL DISCLAIMER")
        report.append("=" * 80)
        report.append("This analysis is generated by an AI system and is NOT a substitute for")
        report.append("professional medical advice, diagnosis, or treatment. Always consult with")
        report.append("a qualified healthcare provider for proper interpretation of your lab")
        report.append("results and medical guidance. Do not make medical decisions based solely")
        report.append("on this automated analysis.")
        report.append("=" * 80)

        return "\n".join(report)

    def _get_status_emoji(self, status: str) -> str:
        """Get emoji for status"""
        emoji_map = {
            'Normal': '✅',
            'High': '⚠️↑',
            'Low': '⚠️↓',
            'Borderline High': '⚡↑',
            'Borderline Low': '⚡↓',
            'Critical High': '🔴↑',
            'Critical Low': '🔴↓'
        }
        return emoji_map.get(status, '•')

    def _format_parameter(self, param: BloodParameter) -> str:
        """Format single parameter for display"""
        emoji = self._get_status_emoji(param.status.value)

        # Calculate deviation percentage
        if param.value < param.reference_min:
            deviation = ((param.reference_min - param.value) / param.reference_min) * 100
            deviation_str = f"({deviation:.1f}% below normal)"
        elif param.value > param.reference_max:
            deviation = ((param.value - param.reference_max) / param.reference_max) * 100
            deviation_str = f"({deviation:.1f}% above normal)"
        else:
            deviation_str = "(within range)"

        return (f"{emoji} {param.name.upper()}: {param.value} {param.unit} "
                f"[Normal: {param.reference_min}-{param.reference_max} {param.unit}] "
                f"{deviation_str}")

print("\n✅ Parameter interpreter ready!")
print("\n")

In [ ]:
print("🎯 BUILDING MULTI-MODEL ORCHESTRATOR")

class BloodReportAnalyzer:
    """Main orchestrator for blood report analysis system"""

    def __init__(self):
        self.parser = UniversalInputParser()
        self.extractor = DataExtractionEngine()
        self.validator = DataValidator()
        self.interpreter = ParameterInterpreter()

    def analyze_report(self, file_path: str) -> Dict:
        """
        Complete end-to-end analysis pipeline

        Args:
            file_path: Path to blood report file

        Returns:
            Dictionary with analysis results
        """
        print()
        print(f"🔬ANALYZING BLOOD REPORT")
        print()

        # Step 1: Parse input file
        print("\n📄 STEP 1: PARSING INPUT FILE")
        print()
        raw_content, parse_success, method = self.parser.parse_file(file_path)

        if not parse_success or not raw_content:
            return {
                'success': False,
                'error': 'Failed to parse input file',
                'report': None,
                'statistics': None
            }

        print(f"   ✅Parsing successful using: {method}")
        print(f"   Extracted {len(raw_content)} characters")

        # Step 2: Extract data
        print("\n🔍 STEP 2: EXTRACTING BLOOD PARAMETERS")
        print()
        extraction_result = self.extractor.extract(raw_content, method)

        if not extraction_result.success:
            return {
                'success': False,
                'error': 'No parameters could be extracted',
                'report': None,
                'statistics': None
            }

        # Step 3: Validate & standardize
        print("\n✅ STEP 3: VALIDATING & STANDARDIZING DATA")
        print()
        validated_result = self.validator.validate_and_standardize(extraction_result)

        # Step 4: Classify parameters
        print("\n STEP 4: CLASSIFYING PARAMETERS (MODEL 1)")
        print()
        classified_result = self.interpreter.classify_parameters(validated_result)

        # Step 5: Generate report
        print("\n STEP 5: GENERATING REPORT")
        print()
        report_text = self.interpreter.generate_report(classified_result)
        print("   ✅ Report generated successfully")

        # Calculate statistics
        stats = self._calculate_statistics(classified_result)

        print()
        print("✅ ANALYSIS COMPLETE")
        print()

        return {
            'success': True,
            'extraction_result': classified_result,
            'report': report_text,
            'statistics': stats
        }

    def _calculate_statistics(self, result: ExtractionResult) -> Dict:
        """Calculate statistics about analysis"""
        stats = {
            'total_parameters': len(result.parameters),
            'normal': 0,
            'abnormal': 0,
            'critical': 0,
            'extraction_method': result.extraction_method,
            'errors': len(result.errors)
        }

        for param in result.parameters:
            if param.status == ParameterStatus.NORMAL:
                stats['normal'] += 1
            elif 'CRITICAL' in param.status.value:
                stats['critical'] += 1
            else:
                stats['abnormal'] += 1

        return stats

# Initialize the analyzer
analyzer = BloodReportAnalyzer()

print("\n Blood Report Analyzer initialized!")

**WEEK 3**

In [ ]:
print("🧩 CELL 10: PATTERN RECOGNITION DATA STRUCTURES")
from dataclasses import dataclass
from typing import List, Dict, Optional
from enum import Enum

class PatternType(Enum):
    """Types of clinical patterns"""
    METABOLIC_SYNDROME = "Metabolic Syndrome"
    DIABETES_RISK = "Diabetes Risk"
    CARDIOVASCULAR_RISK = "Cardiovascular Risk"
    KIDNEY_DISEASE = "Kidney Disease Pattern"
    LIVER_DYSFUNCTION = "Liver Dysfunction Pattern"
    THYROID_DISORDER = "Thyroid Disorder"
    ANEMIA_PATTERN = "Anemia Pattern"
    DYSLIPIDEMIA = "Dyslipidemia Pattern"

class RiskLevel(Enum):
    """Risk assessment levels"""
    LOW = "Low Risk"
    MODERATE = "Moderate Risk"
    HIGH = "High Risk"
    VERY_HIGH = "Very High Risk"

@dataclass
class ClinicalPattern:
    """Represents a detected clinical pattern"""
    pattern_type: PatternType
    detected: bool
    criteria_met: List[str]
    criteria_total: int
    confidence: float  # 0.0 to 1.0
    description: str

@dataclass
class RiskScore:
    """Represents a calculated risk score"""
    score_type: str
    score_value: float
    risk_level: RiskLevel
    percentile: Optional[float]
    interpretation: str

@dataclass
class ContextualFactors:
    """User contextual information"""
    age: Optional[int] = None
    gender: Optional[str] = None  # 'male', 'female'
    smoking: Optional[bool] = None
    diabetes: Optional[bool] = None
    family_history_cvd: Optional[bool] = None

print("\n✅ Pattern recognition data structures configured!")

In [ ]:
print("🔬 CELL 11: METABOLIC SYNDROME DETECTOR")
class MetabolicSyndromeDetector:
    """
    Detects metabolic syndrome using ATP III criteria

    Criteria (need 3+ of 5):
    1. Abdominal obesity (waist circumference) - NOT AVAILABLE IN BLOOD TESTS
    2. Triglycerides ≥ 150 mg/dL
    3. HDL < 40 mg/dL (men) or < 50 mg/dL (women)
    4. Blood pressure ≥ 130/85 mmHg - NOT AVAILABLE IN BLOOD TESTS
    5. Fasting glucose ≥ 100 mg/dL

    Since we only have blood parameters, we detect based on available criteria
    """

    def __init__(self):
        self.criteria = {
            'high_triglycerides': {'threshold': 150, 'unit': 'mg/dL'},
            'low_hdl_male': {'threshold': 40, 'unit': 'mg/dL'},
            'low_hdl_female': {'threshold': 50, 'unit': 'mg/dL'},
            'high_glucose': {'threshold': 100, 'unit': 'mg/dL'},
        }

    def detect(self, parameters: List[BloodParameter], gender: Optional[str] = None) -> ClinicalPattern:
        """
        Detect metabolic syndrome from blood parameters
        """
        criteria_met = []
        criteria_checked = []

        # Get relevant parameters
        triglycerides = self._get_parameter(parameters, 'triglycerides')
        hdl = self._get_parameter(parameters, 'hdl')
        glucose = self._get_parameter(parameters, 'glucose')

        # Check criterion 1: High triglycerides
        if triglycerides:
            criteria_checked.append("Triglycerides")
            if triglycerides.value >= self.criteria['high_triglycerides']['threshold']:
                criteria_met.append(f"High Triglycerides ({triglycerides.value} mg/dL ≥ 150)")

        # Check criterion 2: Low HDL
        if hdl:
            criteria_checked.append("HDL Cholesterol")
            if gender == 'male':
                if hdl.value < self.criteria['low_hdl_male']['threshold']:
                    criteria_met.append(f"Low HDL ({hdl.value} mg/dL < 40 for males)")
            elif gender == 'female':
                if hdl.value < self.criteria['low_hdl_female']['threshold']:
                    criteria_met.append(f"Low HDL ({hdl.value} mg/dL < 50 for females)")
            else:
                # Unknown gender, use more conservative threshold
                if hdl.value < 50:
                    criteria_met.append(f"Low HDL ({hdl.value} mg/dL < 50)")

        # Check criterion 3: High fasting glucose
        if glucose:
            criteria_checked.append("Fasting Glucose")
            if glucose.value >= self.criteria['high_glucose']['threshold']:
                criteria_met.append(f"Elevated Glucose ({glucose.value} mg/dL ≥ 100)")

        # Calculate confidence
        # Need 3 criteria for definite diagnosis, but we can only check 3 from blood
        # If 2/3 are met, suggest possible metabolic syndrome
        # If 3/3 are met, likely metabolic syndrome
        num_criteria_met = len(criteria_met)
        num_criteria_checked = len(criteria_checked)

        detected = num_criteria_met >= 2  # Modified threshold since we can't check all 5

        if num_criteria_checked > 0:
            confidence = num_criteria_met / max(3, num_criteria_checked)  # Normalize to 3 criteria
        else:
            confidence = 0.0

        # Generate description
        if num_criteria_met >= 3:
            description = "Strong indicators of metabolic syndrome detected. All available blood-based criteria are met."
        elif num_criteria_met == 2:
            description = "Possible metabolic syndrome. Two of three available criteria met. Clinical evaluation including waist circumference and blood pressure recommended."
        elif num_criteria_met == 1:
            description = "Some metabolic risk factors present, but does not meet criteria for metabolic syndrome based on blood tests alone."
        else:
            description = "No metabolic syndrome indicators in blood parameters."

        return ClinicalPattern(
            pattern_type=PatternType.METABOLIC_SYNDROME,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=num_criteria_checked,
            confidence=confidence,
            description=description
        )

    def _get_parameter(self, parameters: List[BloodParameter], name: str) -> Optional[BloodParameter]:
        """Helper to get a specific parameter"""
        for param in parameters:
            if param.name == name:
                return param
        return None

print("\n✅ Metabolic syndrome detector ready!")
print("   Implements ATP III criteria (blood-based)")

In [ ]:
print("❤️  CELL 12: CARDIOVASCULAR RISK CALCULATOR")
class CardiovascularRiskCalculator:
    """
    Calculates 10-year cardiovascular disease risk
    Based on simplified Framingham Risk Score

    Risk Factors:
    - Age
    - Gender
    - Total Cholesterol
    - HDL Cholesterol
    - Systolic Blood Pressure (if available)
    - Smoking status (if available)
    - Diabetes status (derived from glucose/HbA1c)
    """

    def __init__(self):
        # Point-based system for risk calculation
        # Simplified version of Framingham scoring
        pass

    def calculate(self, parameters: List[BloodParameter], context: ContextualFactors) -> Optional[RiskScore]:
        """
        Calculate cardiovascular risk score
        """
        if context.age is None or context.gender is None:
            return None

        # Get relevant parameters
        total_chol = self._get_parameter(parameters, 'cholesterol')
        hdl = self._get_parameter(parameters, 'hdl')
        glucose = self._get_parameter(parameters, 'glucose')
        hba1c = self._get_parameter(parameters, 'hba1c')

        # Check if we have minimum required data
        if not total_chol or not hdl:
            return None

        # Determine diabetes status from blood tests if not provided
        has_diabetes = context.diabetes
        if has_diabetes is None:
            has_diabetes = False
            if glucose and glucose.value >= 126:
                has_diabetes = True
            if hba1c and hba1c.value >= 6.5:
                has_diabetes = True

        # Calculate risk using simplified point system
        points = 0

        # Age points (simplified)
        age = context.age
        if context.gender == 'male':
            if age < 35:
                points += -1
            elif age < 45:
                points += 0
            elif age < 55:
                points += 3
            elif age < 65:
                points += 6
            else:
                points += 8
        else:  # female
            if age < 35:
                points += -2
            elif age < 45:
                points += 0
            elif age < 55:
                points += 2
            elif age < 65:
                points += 4
            else:
                points += 6

        # Total cholesterol points
        if total_chol.value < 160:
            points += 0
        elif total_chol.value < 200:
            points += 1
        elif total_chol.value < 240:
            points += 2
        elif total_chol.value < 280:
            points += 3
        else:
            points += 4

        # HDL points (inverse relationship)
        if hdl.value >= 60:
            points += -2
        elif hdl.value >= 50:
            points += -1
        elif hdl.value >= 40:
            points += 0
        elif hdl.value >= 35:
            points += 1
        else:
            points += 2

        # Diabetes points
        if has_diabetes:
            points += 2

        # Smoking points (if available)
        if context.smoking:
            points += 2

        # Convert points to risk percentage
        risk_percentage = self._points_to_risk(points, context.gender)

        # Determine risk level
        if risk_percentage < 10:
            risk_level = RiskLevel.LOW
        elif risk_percentage < 20:
            risk_level = RiskLevel.MODERATE
        elif risk_percentage < 30:
            risk_level = RiskLevel.HIGH
        else:
            risk_level = RiskLevel.VERY_HIGH

        # Generate interpretation
        interpretation = self._generate_interpretation(risk_percentage, risk_level)

        return RiskScore(
            score_type="10-Year Cardiovascular Risk",
            score_value=risk_percentage,
            risk_level=risk_level,
            percentile=None,
            interpretation=interpretation
        )

    def _points_to_risk(self, points: int, gender: str) -> float:
        """
        Convert points to risk percentage
        Simplified mapping based on Framingham data
        """
        # Simplified risk mapping
        if points < 0:
            return 2.0
        elif points < 5:
            return 5.0
        elif points < 10:
            return 10.0
        elif points < 15:
            return 15.0
        elif points < 20:
            return 25.0
        else:
            return 35.0

    def _generate_interpretation(self, risk: float, level: RiskLevel) -> str:
        """Generate risk interpretation"""
        if level == RiskLevel.LOW:
            return f"{risk:.1f}% risk of cardiovascular event in next 10 years. Continue healthy lifestyle habits."
        elif level == RiskLevel.MODERATE:
            return f"{risk:.1f}% risk of cardiovascular event in next 10 years. Lifestyle modifications and regular monitoring recommended."
        elif level == RiskLevel.HIGH:
            return f"{risk:.1f}% risk of cardiovascular event in next 10 years. Intensive risk factor management needed. Consider medication and lifestyle changes."
        else:
            return f"{risk:.1f}% risk of cardiovascular event in next 10 years. High-priority intervention required. Immediate medical consultation recommended."

    def _get_parameter(self, parameters: List[BloodParameter], name: str) -> Optional[BloodParameter]:
        """Helper to get specific parameter"""
        for param in parameters:
            if param.name == name:
                return param
        return None

print("\n✅ Cardiovascular risk calculator ready!")
print("   Implements simplified Framingham Risk Score")

WEEK 4

In [ ]:
print("🔍 CELL 13: CLINICAL PATTERN DETECTOR")
class ClinicalPatternDetector:
    """
    Detects various clinical patterns from blood parameters
    Including: Diabetes risk, kidney disease, liver dysfunction, thyroid disorders, anemia
    """

    def detect_all_patterns(self, parameters: List[BloodParameter], context: Optional[ContextualFactors] = None) -> List[ClinicalPattern]:
        """Detect all recognizable patterns"""
        patterns = []

        patterns.append(self.detect_diabetes_risk(parameters))
        patterns.append(self.detect_kidney_disease(parameters))
        patterns.append(self.detect_liver_dysfunction(parameters))
        patterns.append(self.detect_thyroid_disorder(parameters))
        patterns.append(self.detect_anemia_pattern(parameters))
        patterns.append(self.detect_dyslipidemia(parameters, context))

        # Filter to only detected patterns
        return [p for p in patterns if p.detected]

    def detect_diabetes_risk(self, parameters: List[BloodParameter]) -> ClinicalPattern:
        """Detect diabetes or prediabetes"""
        criteria_met = []

        glucose = self._get_param(parameters, 'glucose')
        hba1c = self._get_param(parameters, 'hba1c')

        if glucose:
            if glucose.value >= 126:
                criteria_met.append(f"Fasting glucose {glucose.value} mg/dL (≥126 = Diabetic range)")
            elif glucose.value >= 100:
                criteria_met.append(f"Fasting glucose {glucose.value} mg/dL (100-125 = Prediabetic range)")

        if hba1c:
            if hba1c.value >= 6.5:
                criteria_met.append(f"HbA1c {hba1c.value}% (≥6.5 = Diabetic range)")
            elif hba1c.value >= 5.7:
                criteria_met.append(f"HbA1c {hba1c.value}% (5.7-6.4 = Prediabetic range)")

        detected = len(criteria_met) > 0
        confidence = 1.0 if len(criteria_met) >= 2 else 0.7 if len(criteria_met) == 1 else 0.0

        if detected:
            if any('Diabetic range' in c for c in criteria_met):
                description = "Diabetes detected based on glucose/HbA1c levels. Medical consultation required."
            else:
                description = "Prediabetes detected. Lifestyle modifications and monitoring recommended."
        else:
            description = "No diabetes risk indicators detected."

        return ClinicalPattern(
            pattern_type=PatternType.DIABETES_RISK,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=2,
            confidence=confidence,
            description=description
        )

    def detect_kidney_disease(self, parameters: List[BloodParameter]) -> ClinicalPattern:
        """Detect kidney disease pattern"""
        criteria_met = []

        creatinine = self._get_param(parameters, 'creatinine')
        bun = self._get_param(parameters, 'bun')
        uric_acid = self._get_param(parameters, 'uric_acid')

        if creatinine and creatinine.value > 1.3:
            criteria_met.append(f"Elevated creatinine ({creatinine.value} mg/dL > 1.3)")

        if bun and bun.value > 20:
            criteria_met.append(f"Elevated BUN ({bun.value} mg/dL > 20)")

        if uric_acid and uric_acid.value > 7.0:
            criteria_met.append(f"Elevated uric acid ({uric_acid.value} mg/dL > 7.0)")

        # Calculate BUN/Creatinine ratio if both available
        if creatinine and bun:
            ratio = bun.value / creatinine.value
            if ratio > 20:
                criteria_met.append(f"Elevated BUN/Creatinine ratio ({ratio:.1f} > 20)")

        detected = len(criteria_met) >= 2
        confidence = min(len(criteria_met) / 3, 1.0)

        if detected:
            description = "Kidney function abnormalities detected. Nephrology consultation recommended."
        else:
            description = "No significant kidney disease pattern detected."

        return ClinicalPattern(
            pattern_type=PatternType.KIDNEY_DISEASE,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=4,
            confidence=confidence,
            description=description
        )

    def detect_liver_dysfunction(self, parameters: List[BloodParameter]) -> ClinicalPattern:
        """Detect liver dysfunction pattern"""
        criteria_met = []

        alt = self._get_param(parameters, 'alt')
        ast = self._get_param(parameters, 'ast')
        alp = self._get_param(parameters, 'alp')
        bilirubin = self._get_param(parameters, 'bilirubin_total')
        albumin = self._get_param(parameters, 'albumin')

        if alt and alt.value > 56:
            criteria_met.append(f"Elevated ALT ({alt.value} U/L > 56)")

        if ast and ast.value > 40:
            criteria_met.append(f"Elevated AST ({ast.value} U/L > 40)")

        if alp and alp.value > 147:
            criteria_met.append(f"Elevated ALP ({alp.value} U/L > 147)")

        if bilirubin and bilirubin.value > 1.2:
            criteria_met.append(f"Elevated bilirubin ({bilirubin.value} mg/dL > 1.2)")

        if albumin and albumin.value < 3.5:
            criteria_met.append(f"Low albumin ({albumin.value} g/dL < 3.5)")

        # Check AST/ALT ratio if both available
        if ast and alt:
            ratio = ast.value / alt.value
            if ratio > 2:
                criteria_met.append(f"AST/ALT ratio elevated ({ratio:.2f} > 2, suggests alcoholic liver disease)")

        detected = len(criteria_met) >= 2
        confidence = min(len(criteria_met) / 4, 1.0)

        if detected:
            description = "Liver function abnormalities detected. Hepatology evaluation recommended."
        else:
            description = "No significant liver dysfunction pattern detected."

        return ClinicalPattern(
            pattern_type=PatternType.LIVER_DYSFUNCTION,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=6,
            confidence=confidence,
            description=description
        )

    def detect_thyroid_disorder(self, parameters: List[BloodParameter]) -> ClinicalPattern:
        """Detect thyroid disorders"""
        criteria_met = []

        tsh = self._get_param(parameters, 'tsh')
        t3 = self._get_param(parameters, 't3')
        t4 = self._get_param(parameters, 't4')

        disorder_type = None

        if tsh:
            if tsh.value > 4.0:
                criteria_met.append(f"Elevated TSH ({tsh.value} mIU/L > 4.0)")
                disorder_type = "Hypothyroidism"
            elif tsh.value < 0.4:
                criteria_met.append(f"Low TSH ({tsh.value} mIU/L < 0.4)")
                disorder_type = "Hyperthyroidism"

        if t3:
            if t3.value < 80:
                criteria_met.append(f"Low T3 ({t3.value} ng/dL < 80)")
            elif t3.value > 200:
                criteria_met.append(f"High T3 ({t3.value} ng/dL > 200)")

        if t4:
            if t4.value < 5.0:
                criteria_met.append(f"Low T4 ({t4.value} µg/dL < 5.0)")
            elif t4.value > 12.0:
                criteria_met.append(f"High T4 ({t4.value} µg/dL > 12.0)")

        detected = len(criteria_met) > 0
        confidence = min(len(criteria_met) / 2, 1.0)

        if detected:
            if disorder_type:
                description = f"Pattern suggestive of {disorder_type}. Endocrinology consultation recommended."
            else:
                description = "Thyroid function abnormalities detected. Further thyroid evaluation needed."
        else:
            description = "No thyroid disorder pattern detected."

        return ClinicalPattern(
            pattern_type=PatternType.THYROID_DISORDER,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=3,
            confidence=confidence,
            description=description
        )

    def detect_anemia_pattern(self, parameters: List[BloodParameter]) -> ClinicalPattern:
        """Detect anemia and classify type"""
        criteria_met = []

        hb = self._get_param(parameters, 'hemoglobin')
        mcv = self._get_param(parameters, 'mcv')
        mch = self._get_param(parameters, 'mch')

        anemia_type = None

        if hb:
            if hb.value < 12.0:  # Using female lower limit as more conservative
                criteria_met.append(f"Low hemoglobin ({hb.value} g/dL < 12.0)")

                # Classify anemia type based on MCV
                if mcv:
                    if mcv.value < 80:
                        anemia_type = "Microcytic anemia (possibly iron deficiency)"
                        criteria_met.append(f"Low MCV ({mcv.value} fL < 80)")
                    elif mcv.value > 100:
                        anemia_type = "Macrocytic anemia (possibly B12/folate deficiency)"
                        criteria_met.append(f"High MCV ({mcv.value} fL > 100)")
                    else:
                        anemia_type = "Normocytic anemia"

        detected = len(criteria_met) > 0
        confidence = 1.0 if anemia_type else 0.7

        if detected:
            if anemia_type:
                description = f"{anemia_type} detected. Hematology evaluation and specific testing recommended."
            else:
                description = "Anemia detected. Further evaluation needed to determine cause."
        else:
            description = "No anemia pattern detected."

        return ClinicalPattern(
            pattern_type=PatternType.ANEMIA_PATTERN,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=3,
            confidence=confidence,
            description=description
        )

    def detect_dyslipidemia(self, parameters: List[BloodParameter], context: Optional[ContextualFactors] = None) -> ClinicalPattern:
        """Detect atherogenic dyslipidemia"""
        criteria_met = []

        total_chol = self._get_param(parameters, 'cholesterol')
        ldl = self._get_param(parameters, 'ldl')
        hdl = self._get_param(parameters, 'hdl')
        trig = self._get_param(parameters, 'triglycerides')

        if total_chol and total_chol.value > 200:
            criteria_met.append(f"High total cholesterol ({total_chol.value} mg/dL > 200)")

        if ldl and ldl.value > 100:
            severity = "Very high" if ldl.value > 190 else "High"
            criteria_met.append(f"{severity} LDL ({ldl.value} mg/dL > 100)")

        if hdl:
            threshold = 40 if (context and context.gender == 'male') else 50
            if hdl.value < threshold:
                criteria_met.append(f"Low HDL ({hdl.value} mg/dL < {threshold})")

        if trig and trig.value > 150:
            severity = "Very high" if trig.value > 500 else "High"
            criteria_met.append(f"{severity} triglycerides ({trig.value} mg/dL > 150)")

        # Calculate ratios
        if total_chol and hdl:
            ratio = total_chol.value / hdl.value
            if ratio > 5.0:
                criteria_met.append(f"Elevated TC/HDL ratio ({ratio:.2f} > 5.0)")

        detected = len(criteria_met) >= 2
        confidence = min(len(criteria_met) / 4, 1.0)

        if detected:
            description = "Atherogenic dyslipidemia detected. Lipid management and cardiovascular risk assessment recommended."
        else:
            description = "No significant dyslipidemia pattern detected."

        return ClinicalPattern(
            pattern_type=PatternType.DYSLIPIDEMIA,
            detected=detected,
            criteria_met=criteria_met,
            criteria_total=5,
            confidence=confidence,
            description=description
        )

    def _get_param(self, parameters: List[BloodParameter], name: str) -> Optional[BloodParameter]:
        """Helper to get specific parameter"""
        for param in parameters:
            if param.name == name:
                return param
        return None

print("\n✅ Clinical pattern detector ready!")
print("   Detects: Diabetes, Kidney, Liver, Thyroid, Anemia, Dyslipidemia")

In [ ]:
print("👤 CELL 14: CONTEXTUAL ANALYZER (MODEL 3)")
class ContextualAnalyzer:
    """
    Model 3: Provides context-aware analysis
    Adjusts reference ranges and interpretations based on age/gender
    """

    def adjust_reference_ranges(self, parameters: List[BloodParameter], context: ContextualFactors) -> List[BloodParameter]:
        """
        Adjust reference ranges based on context (age/gender)
        Returns new list with adjusted ranges
        """
        adjusted_params = []

        for param in parameters:
            adjusted_param = BloodParameter(
                name=param.name,
                value=param.value,
                unit=param.unit,
                reference_min=param.reference_min,
                reference_max=param.reference_max,
                status=param.status,
                raw_text=param.raw_text
            )

            # Adjust hemoglobin for gender
            if param.name == 'hemoglobin' and context.gender:
                if context.gender == 'male':
                    adjusted_param.reference_min = 13.5
                    adjusted_param.reference_max = 17.5
                else:
                    adjusted_param.reference_min = 12.0
                    adjusted_param.reference_max = 15.5

            # Adjust HDL for gender
            elif param.name == 'hdl' and context.gender:
                if context.gender == 'male':
                    adjusted_param.reference_min = 40
                else:
                    adjusted_param.reference_min = 50
                adjusted_param.reference_max = 999

            # Adjust creatinine for gender
            elif param.name == 'creatinine' and context.gender:
                if context.gender == 'male':
                    adjusted_param.reference_min = 0.7
                    adjusted_param.reference_max = 1.3
                else:
                    adjusted_param.reference_min = 0.6
                    adjusted_param.reference_max = 1.1

            adjusted_params.append(adjusted_param)

        return adjusted_params

    def generate_contextual_insights(self, parameters: List[BloodParameter],
                                    patterns: List[ClinicalPattern],
                                    context: ContextualFactors) -> List[str]:
        """
        Generate context-specific insights and recommendations
        """
        insights = []

        # Age-based insights
        if context.age:
            if context.age >= 65:
                insights.append(f"Age-related consideration: At {context.age} years, cardiovascular and kidney function monitoring is particularly important.")
            elif context.age >= 50:
                insights.append(f"Age-related consideration: At {context.age} years, regular screening for diabetes and cardiovascular disease is recommended.")
            elif context.age < 40:
                insights.append(f"Age consideration: At {context.age} years, focus on preventive health and establishing healthy baseline values.")

        # Gender-specific insights
        if context.gender == 'female':
            # Check for anemia
            hb = self._get_param(parameters, 'hemoglobin')
            if hb and hb.value < 12:
                insights.append("Gender consideration: Lower hemoglobin may be related to menstruation. Iron supplementation may be beneficial.")

        # Pattern-specific contextual insights
        for pattern in patterns:
            if pattern.pattern_type == PatternType.METABOLIC_SYNDROME and context.age:
                if context.age >= 40:
                    insights.append("Metabolic syndrome at your age significantly increases cardiovascular risk. Aggressive lifestyle modification is crucial.")

            elif pattern.pattern_type == PatternType.DIABETES_RISK and context.family_history_cvd:
                insights.append("Family history of cardiovascular disease combined with diabetes risk requires intensive monitoring and prevention strategies.")

        return insights

    def _get_param(self, parameters: List[BloodParameter], name: str) -> Optional[BloodParameter]:
        """Helper to get specific parameter"""
        for param in parameters:
            if param.name == name:
                return param
        return None

print("\n✅ Contextual analyzer ready!")
print("   Adjusts ranges for age/gender, provides personalized insights")

In [ ]:
print("🎯 CELL 15: INTEGRATED MULTI-MODEL ANALYZER")

class IntegratedBloodAnalyzer:
    """
    Integrated analyzer combining all three models:
    - Model 1: Parameter Interpretation (from Week 1)
    - Model 2: Pattern Recognition & Risk Assessment
    - Model 3: Contextual Analysis
    """

    def __init__(self):
        # Model 2 components
        self.metabolic_detector = MetabolicSyndromeDetector()
        self.cv_risk_calculator = CardiovascularRiskCalculator()
        self.pattern_detector = ClinicalPatternDetector()

        # Model 3 component
        self.contextual_analyzer = ContextualAnalyzer()

    def comprehensive_analysis(self, parameters: List[BloodParameter],
                              context: Optional[ContextualFactors] = None) -> Dict:
        """
        Perform comprehensive multi-model analysis

        Returns:
            Dictionary containing:
            - patterns: List of detected clinical patterns
            - risk_scores: List of calculated risk scores
            - contextual_insights: Context-specific recommendations
            - overall_assessment: Summary assessment
        """
        print("\n   🔬 Running Model 2: Pattern Recognition...")

        # Model 2: Detect clinical patterns
        patterns = []

        # Metabolic syndrome
        metabolic_pattern = self.metabolic_detector.detect(
            parameters,
            context.gender if context else None
        )
        if metabolic_pattern.detected or metabolic_pattern.confidence > 0:
            patterns.append(metabolic_pattern)

        # Other clinical patterns
        other_patterns = self.pattern_detector.detect_all_patterns(parameters, context)
        patterns.extend(other_patterns)

        print(f"✅ Detected {len(patterns)} clinical pattern(s)")

        # Model 2: Calculate risk scores
        print("\n📊 Calculating risk scores...")
        risk_scores = []

        if context:
            cv_risk = self.cv_risk_calculator.calculate(parameters, context)
            if cv_risk:
                risk_scores.append(cv_risk)
                print(f"✅ Cardiovascular risk: {cv_risk.risk_level.value}")

        # Model 3: Contextual analysis
        contextual_insights = []
        if context:
            print("\n   👤 Running Model 3: Contextual Analysis...")
            contextual_insights = self.contextual_analyzer.generate_contextual_insights(
                parameters, patterns, context
            )
            print(f"      ✅ Generated {len(contextual_insights)} contextual insight(s)")

        # Generate overall assessment
        overall_assessment = self._generate_overall_assessment(patterns, risk_scores, context)

        return {
            'patterns': patterns,
            'risk_scores': risk_scores,
            'contextual_insights': contextual_insights,
            'overall_assessment': overall_assessment
        }

    def _generate_overall_assessment(self, patterns: List[ClinicalPattern],
                                    risk_scores: List[RiskScore],
                                    context: Optional[ContextualFactors]) -> str:
        """Generate overall health assessment"""

        if not patterns and not risk_scores:
            return "No significant clinical patterns or elevated risk factors detected. Continue regular health monitoring."

        assessment_parts = []

        # High-priority patterns
        critical_patterns = [p for p in patterns if p.confidence >= 0.8]
        if critical_patterns:
            assessment_parts.append(f"PRIORITY: {len(critical_patterns)} significant clinical pattern(s) detected requiring medical attention.")

        # Risk assessment
        high_risks = [r for r in risk_scores if r.risk_level in [RiskLevel.HIGH, RiskLevel.VERY_HIGH]]
        if high_risks:
            assessment_parts.append(f"RISK ALERT: Elevated cardiovascular risk detected. Intervention recommended.")

        # Moderate concerns
        moderate_patterns = [p for p in patterns if 0.5 <= p.confidence < 0.8]
        if moderate_patterns:
            assessment_parts.append(f"{len(moderate_patterns)} potential health concern(s) identified. Monitoring and lifestyle modifications advised.")

        if not assessment_parts:
            assessment_parts.append("Some health indicators noted. Regular follow-up recommended.")

        return " ".join(assessment_parts)

print("\n✅ Integrated multi-model analyzer ready!")
print("   Combines Models 1, 2, and 3 for comprehensive analysis")

In [ ]:
print("📋 CELL 16: ENHANCED REPORT GENERATOR")
class EnhancedReportGenerator:

    def generate_comprehensive_report(self,
                                     extraction_result: ExtractionResult,
                                     analysis_result: Dict,
                                     context: Optional[ContextualFactors] = None) -> str:
        """Generate comprehensive multi-model report"""

        report = []
        report.append()
        report.append("COMPREHENSIVE BLOOD REPORT ANALYSIS")
        report.append("Multi-Model AI System (Models 1, 2, 3)")
        report.append()
        report.append("")

        # Context information
        if context:
            report.append("📋 PATIENT CONTEXT")
            report.append()
            if context.age:
                report.append(f"Age: {context.age} years")
            if context.gender:
                report.append(f"Gender: {context.gender.title()}")
            if context.smoking is not None:
                report.append(f"Smoking Status: {'Yes' if context.smoking else 'No'}")
            if context.diabetes is not None:
                report.append(f"Known Diabetes: {'Yes' if context.diabetes else 'No'}")
            report.append("")
            report.append()
            report.append("")

        # Overall Assessment (Executive Summary)
        report.append("🎯 OVERALL HEALTH ASSESSMENT")
        report.append()
        report.append(analysis_result['overall_assessment'])
        report.append("")
        report.append()
        report.append("")

        # Clinical Patterns (Model 2)
        patterns = analysis_result['patterns']
        if patterns:
            report.append("🔍 CLINICAL PATTERNS DETECTED (Model 2: Pattern Recognition)")
            report.append()
            report.append(f"Total Patterns Identified: {len(patterns)}")
            report.append("")

            for i, pattern in enumerate(patterns, 1):
                confidence_pct = pattern.confidence * 100
                report.append(f"{i}. {pattern.pattern_type.value}")
                report.append(f"   Confidence: {confidence_pct:.0f}%")
                report.append(f"   Criteria Met: {len(pattern.criteria_met)}/{pattern.criteria_total}")

                if pattern.criteria_met:
                    report.append(f"   Findings:")
                    for criterion in pattern.criteria_met:
                        report.append(f"      • {criterion}")

                report.append(f"   Assessment: {pattern.description}")
                report.append("")

            report.append()

        # Risk Scores (Model 2)
        risk_scores = analysis_result['risk_scores']
        if risk_scores:
            report.append("📊 RISK ASSESSMENT (Model 2: Risk Calculation)")
            report.append()

            for risk in risk_scores:
                report.append(f"🫀 {risk.score_type}")
                report.append(f"   Risk Score: {risk.score_value:.1f}%")
                report.append(f"   Risk Level: {risk.risk_level.value}")
                report.append(f"   Interpretation: {risk.interpretation}")
                report.append("")

            report.append()
        # Contextual Insights (Model 3)
        contextual_insights = analysis_result.get('contextual_insights', [])
        if contextual_insights:
            report.append("👤 PERSONALIZED INSIGHTS (Model 3: Contextual Analysis)")
            report.append()

            for i, insight in enumerate(contextual_insights, 1):
                report.append(f"{i}. {insight}")

            report.append()

        # Individual Parameters (Model 1) - Summary
        report.append("🧪 INDIVIDUAL PARAMETER RESULTS (Model 1: Interpretation)")
        report.append()

        # Count by status
        status_counts = {}
        for param in extraction_result.parameters:
            status = param.status.value
            status_counts[status] = status_counts.get(status, 0) + 1

        report.append(f"Total Parameters Analyzed: {len(extraction_result.parameters)}")
        report.append("")
        for status, count in sorted(status_counts.items()):
            emoji = self._get_status_emoji(status)
            report.append(f"   {emoji} {status}: {count}")
        report.append("")
        report.append()
        report.append("")

        # Group parameters
        critical = [p for p in extraction_result.parameters if 'CRITICAL' in p.status.value]
        abnormal = [p for p in extraction_result.parameters
                   if p.status not in [ParameterStatus.NORMAL] and 'CRITICAL' not in p.status.value]
        normal = [p for p in extraction_result.parameters if p.status == ParameterStatus.NORMAL]

        # Critical values
        if critical:
            report.append("🔴 CRITICAL VALUES")
            report.append()
            for param in critical:
                report.append(self._format_parameter(param))
            report.append("")

        # Abnormal values
        if abnormal:
            report.append("⚠️  ABNORMAL VALUES")
            report.append()
            for param in abnormal:
                report.append(self._format_parameter(param))
            report.append("")

        # Normal values (summarized)
        if normal:
            report.append(f"✅ NORMAL VALUES ({len(normal)} parameters)")
            report.append()
            normal_names = [p.name.upper() for p in normal]
            report.append(f"   {', '.join(normal_names)}")
            report.append("")

        report.append()
        # Recommendations
        report.append("💡 RECOMMENDATIONS")
        report.append()
        recommendations = self._generate_recommendations(patterns, risk_scores, critical, abnormal)
        for rec in recommendations:
            report.append(f"• {rec}")
        report.append()

        # Disclaimer
        report.append("⚕️  MEDICAL DISCLAIMER")
        report.append()
        report.append("This comprehensive analysis is generated by an AI system using multiple")
        report.append("analytical models and is NOT a substitute for professional medical advice,")
        report.append("diagnosis, or treatment. Pattern detection and risk calculations are based")
        report.append("on statistical models and clinical guidelines but require validation by")
        report.append("qualified healthcare providers. Always consult with your doctor for proper")
        report.append("interpretation and medical guidance.")
        report.append()

        return "\n".join(report)

    def _get_status_emoji(self, status: str) -> str:
        """Get emoji for status"""
        emoji_map = {
            'Normal': '✅',
            'High': '⚠️↑',
            'Low': '⚠️↓',
            'Borderline High': '⚡↑',
            'Borderline Low': '⚡↓',
            'Critical High': '🔴↑',
            'Critical Low': '🔴↓'
        }
        return emoji_map.get(status, '•')

    def _format_parameter(self, param: BloodParameter) -> str:
        """Format single parameter"""
        emoji = self._get_status_emoji(param.status.value)
        return (f"{emoji} {param.name.upper()}: {param.value} {param.unit} "
                f"[Normal: {param.reference_min}-{param.reference_max} {param.unit}]")

    def _generate_recommendations(self, patterns: List[ClinicalPattern],
                                 risk_scores: List[RiskScore],
                                 critical: List[BloodParameter],
                                 abnormal: List[BloodParameter]) -> List[str]:
        """Generate actionable recommendations"""
        recommendations = []

        if critical:
            recommendations.append("URGENT: Seek immediate medical attention for critical values.")

        if any(p.pattern_type == PatternType.METABOLIC_SYNDROME for p in patterns):
            recommendations.append("Address metabolic syndrome through weight management, diet, and exercise.")

        if any(p.pattern_type == PatternType.DIABETES_RISK for p in patterns):
            recommendations.append("Implement diabetes prevention strategies: regular exercise, low-glycemic diet, weight control.")

        if any(r.risk_level in [RiskLevel.HIGH, RiskLevel.VERY_HIGH] for r in risk_scores):
            recommendations.append("High cardiovascular risk: Consider statin therapy, blood pressure management, and lifestyle modifications.")

        if any(p.pattern_type == PatternType.DYSLIPIDEMIA for p in patterns):
            recommendations.append("Lipid management: Reduce saturated fats, increase omega-3 fatty acids, consider medication if lifestyle changes insufficient.")

        if abnormal and not critical:
            recommendations.append("Schedule follow-up with healthcare provider to discuss abnormal values.")

        if not recommendations:
            recommendations.append("Continue healthy lifestyle habits and regular health monitoring.")

        return recommendations

print("\n✅ Enhanced report generator ready!")
print("   Generates comprehensive multi-model reports")

In [ ]:
print("🔄 CELL 17: UPDATING MAIN ANALYZER")
print("=" * 80)

# Enhance the existing BloodReportAnalyzer class
class EnhancedBloodReportAnalyzer(BloodReportAnalyzer):
    """
    Enhanced analyzer with Models 1, 2, and 3
    Extends the Week 1 analyzer with pattern recognition and risk assessment
    """

    def __init__(self):
        super().__init__()
        # Add Model 2 & 3 components
        self.integrated_analyzer = IntegratedBloodAnalyzer()
        self.enhanced_reporter = EnhancedReportGenerator()

    def analyze_report_comprehensive(self, file_path: str,
                                    context: Optional[ContextualFactors] = None) -> Dict:
        """
        Complete comprehensive analysis with all three models

        Args:
            file_path: Path to blood report
            context: Optional patient context (age, gender, etc.)

        Returns:
            Dictionary with complete analysis results
        """
        print(f"\n{'=' * 80}")
        print(f"🔬 COMPREHENSIVE MULTI-MODEL ANALYSIS")
        print(f"{'=' * 80}")

        # Step 1-4: Run Week 1 pipeline (Models 1)
        basic_result = self.analyze_report(file_path)

        if not basic_result['success']:
            return basic_result

        classified_result = basic_result['extraction_result']

        # Step 5: Run Models 2 & 3 analysis
        print("\n" + "=" * 80)
        print("🎯 RUNNING ADVANCED ANALYSIS (Models 2 & 3)")
        print("=" * 80)

        advanced_analysis = self.integrated_analyzer.comprehensive_analysis(
            classified_result.parameters,
            context
        )

        # Step 6: Generate comprehensive report
        print("\n📋 GENERATING COMPREHENSIVE REPORT")
        print("-" * 80)

        comprehensive_report = self.enhanced_reporter.generate_comprehensive_report(
            classified_result,
            advanced_analysis,
            context
        )

        print("✅ Comprehensive report generated")

        # Calculate enhanced statistics
        stats = basic_result['statistics'].copy()
        stats['patterns_detected'] = len(advanced_analysis['patterns'])
        stats['risk_scores_calculated'] = len(advanced_analysis['risk_scores'])
        stats['contextual_insights'] = len(advanced_analysis.get('contextual_insights', []))

        print("\n" + "=" * 80)
        print("✅ COMPREHENSIVE ANALYSIS COMPLETE")
        print("=" * 80)

        return {
            'success': True,
            'extraction_result': classified_result,
            'advanced_analysis': advanced_analysis,
            'report': comprehensive_report,
            'statistics': stats,
            'context': context
        }
analyzer = EnhancedBloodReportAnalyzer()

print("\n✅ Main analyzer updated with Models 2 & 3!")
print("   Now supports comprehensive multi-model analysis")

In [ ]:
print("📤 ENHANCED BLOOD REPORT ANALYSIS WITH CONTEXT")
print()
print("\n🎯 This interface supports:")
print("   • All file formats (PDF, JSON, Images, Text)")
print("   • Patient context (Age, Gender, Medical History)")
print("   • Multi-model analysis (Models 1, 2, 3)")
print("   • Pattern detection & Risk assessment")
print()

# Step 1: Collect patient context
print("\n📋 STEP 1: PATIENT CONTEXT (Optional but Recommended)")
print()
print("Providing context enables Models 2 & 3 for advanced analysis:")
print("  • Model 2: Pattern recognition & Risk scores")
print("  • Model 3: Personalized interpretations")
print("\nLeave blank to skip any field.")
print()

# Collect context
try:
    age_input = input("Patient Age (years): ").strip()
    age = int(age_input) if age_input else None
except:
    age = None

gender_input = input("Patient Gender (male/female): ").strip().lower()
gender = gender_input if gender_input in ['male', 'female'] else None

smoking_input = input("Smoking Status (yes/no): ").strip().lower()
smoking = True if smoking_input == 'yes' else False if smoking_input == 'no' else None

diabetes_input = input("Known Diabetes (yes/no): ").strip().lower()
diabetes = True if diabetes_input == 'yes' else False if diabetes_input == 'no' else None

family_cvd_input = input("Family History of Heart Disease (yes/no): ").strip().lower()
family_history_cvd = True if family_cvd_input == 'yes' else False if family_cvd_input == 'no' else None

# Create context object
context = ContextualFactors(
    age=age,
    gender=gender,
    smoking=smoking,
    diabetes=diabetes,
    family_history_cvd=family_history_cvd
)

print("\n✅ Context captured!")
if age:
    print(f"   Age: {age} years")
if gender:
    print(f"   Gender: {gender.title()}")
if smoking is not None:
    print(f"   Smoking: {'Yes' if smoking else 'No'}")

# Step 2: Upload file
print()
print("📤 STEP 2: UPLOAD BLOOD REPORT")
print()
print("Click 'Choose Files' to upload your report...")
print()

from google.colab import files
uploaded = files.upload()

if not uploaded:
    print("\n❌ No file uploaded.")
else:
    filename = list(uploaded.keys())[0]

    print()
    print(f"📄 FILE RECEIVED")
    print()
    print(f"   Filename: {filename}")
    print(f"   Size: {len(uploaded[filename]) / 1024:.2f} KB")

    try:
        # Run comprehensive analysis
        print()
        print(f"STARTING COMPREHENSIVE ANALYSIS")
        print()

        result = analyzer.analyze_report_comprehensive(filename, context)

        if result['success']:
            # Display report
            print("\n\n")
            print(result['report'])

            # Display statistics
            print()
            print("📊 ANALYSIS STATISTICS")
            print()
            stats = result['statistics']

            print(f"\n📋 Model 1 (Parameter Interpretation):")
            print(f"   • Parameters Extracted: {stats['total_parameters']}")
            print(f"   • Normal: {stats['normal']}")
            print(f"   • Abnormal: {stats['abnormal']}")
            print(f"   • Critical: {stats['critical']}")

            print(f"\n🔍 Model 2 (Pattern Recognition & Risk Assessment):")
            print(f"   • Clinical Patterns Detected: {stats['patterns_detected']}")
            print(f"   • Risk Scores Calculated: {stats['risk_scores_calculated']}")

            if context.age or context.gender:
                print(f"\n👤 Model 3 (Contextual Analysis):")
                print(f"   • Personalized Insights: {stats['contextual_insights']}")
                print(f"   • Context-Adjusted Interpretations: Enabled")
            else:
                print(f"\n👤 Model 3 (Contextual Analysis):")
                print(f"   • Status: Not used (no context provided)")

            # Pattern details
            if stats['patterns_detected'] > 0:
                print(f"\n🧩 Detected Patterns:")
                for pattern in result['advanced_analysis']['patterns']:
                    emoji = "🔴" if pattern.confidence > 0.8 else "⚠️"
                    print(f"   {emoji} {pattern.pattern_type.value} (Confidence: {pattern.confidence*100:.0f}%)")

            # Risk scores
            if stats['risk_scores_calculated'] > 0:
                print(f"\n📈 Risk Assessments:")
                for risk in result['advanced_analysis']['risk_scores']:
                    emoji = "🔴" if risk.risk_level in [RiskLevel.HIGH, RiskLevel.VERY_HIGH] else "⚠️" if risk.risk_level == RiskLevel.MODERATE else "✅"
                    print(f"   {emoji} {risk.score_type}: {risk.score_value:.1f}% ({risk.risk_level.value})")

            # Save report
            print()
            print("💾 SAVING REPORT")
            print()

            base_name = filename.rsplit('.', 1)[0]
            report_filename = f"comprehensive_analysis_{base_name}.txt"

            with open(report_filename, 'w', encoding='utf-8') as f:
                f.write(result['report'])

                # Add detailed statistics
                f.write("\n\n")
                f.write("DETAILED ANALYSIS STATISTICS\n")
                f.write("\n")
                f.write(f"Model 1 - Parameters: {stats['total_parameters']}\n")
                f.write(f"Model 2 - Patterns: {stats['patterns_detected']}\n")
                f.write(f"Model 2 - Risk Scores: {stats['risk_scores_calculated']}\n")
                f.write(f"Model 3 - Insights: {stats['contextual_insights']}\n")

            print(f"   ✅ Report saved as: {report_filename}")

            # Download
            print("\n📥 Downloading report...")
            try:
                files.download(report_filename)
                print("   ✅ Download initiated!")
            except:
                print("   ℹ️  File saved in Colab workspace")

            # Summary
            print("\n\n")
            print("COMPREHENSIVE ANALYSIS COMPLETE!")
            print()

            if stats['critical'] > 0:
                print(f"\n🔴 URGENT: {stats['critical']} CRITICAL VALUE(S) DETECTED!")
                print("   Immediate medical attention recommended.")
            elif stats['patterns_detected'] > 0:
                print(f"\n⚠️  {stats['patterns_detected']} CLINICAL PATTERN(S) IDENTIFIED")
                print("   Review the detailed report and consult your healthcare provider.")
            elif stats['abnormal'] > 0:
                print(f"\n⚠️  {stats['abnormal']} ABNORMAL VALUE(S) DETECTED")
                print("   Consider scheduling a follow-up with your doctor.")
            else:
                print("\n✅ ALL PARAMETERS WITHIN NORMAL RANGE")
                print("   Continue regular health monitoring.")

        else:
            print("\n❌ ANALYSIS FAILED")
            print(f"Error: {result.get('error', 'Unknown error')}")

    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()

print("\n\n")
print("🔄 RUN THIS CELL AGAIN TO ANALYZE ANOTHER REPORT")